[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Build de contenedores en Actions.

Py271 formaliza esta parte: el artefacto deja de ser solo un paquete descargable y pasa a ser una imagen versionada, trazable y lista para promotion.

## Objetivos.

- Entender el flujo checkout -> build -> push.
- Revisar tags inmutables.
- Preparar la integracion con registros de imagenes.

## Del checkout al registro.

Un pipeline de build de contenedores suele incluir estos pasos: checkout, autenticacion al registro, `docker buildx build`, etiquetado inmutable y push hacia GHCR o Artifact Registry.

En este bloque todavia no desplegamos: construimos y publicamos un artefacto listo para ser validado, escaneado y promovido.

## Ejemplo conceptual: Actions dispara Cloud Build.

```yaml
- name: Authenticate to GCP
  uses: google-github-actions/auth@v2
  with:
    workload_identity_provider: ${{ secrets.GCP_WIF_PROVIDER }}
    service_account: cicd-builder@mi-proyecto.iam.gserviceaccount.com

- name: Submit build
  run: gcloud builds submit --tag us-central1-docker.pkg.dev/mi-proyecto/apps/api:${{ github.sha }} .
```

Este patron mantiene el control de flujo en GitHub Actions, pero delega la construccion de la imagen a Google Cloud Build.

## Google Cloud Build en la estrategia de construccion.

GitHub Actions no es la unica opcion para construir imagenes. En entornos GCP, Google Cloud Build puede actuar como servicio gestionado de build, ya sea disparado desde GitHub Actions o mediante triggers propios conectados al repositorio.

Eso resulta util cuando se quiere:

- construir dentro del perimetro operativo de GCP;
- reducir logica de build en runners de GitHub;
- integrar de forma nativa Artifact Registry, cuentas de servicio y auditoria de GCP;
- separar orquestacion del pipeline y ejecucion del build.

En otras palabras: GitHub Actions puede coordinar el flujo y Cloud Build puede ejecutar la construccion.

## Endurecimiento en el flujo de build.

Publicar una imagen sin validarla es equivalente a hacer merge sin pasar los tests: el artefacto llega al registro con problemas que podrian haberse detectado antes. El endurecimiento de contenedores agrega dos pasos al flujo de build:

```text
checkout
    |
    v
Hadolint          <- analisis estatico del Dockerfile (antes del build)
    |
    v
docker build      <- construccion de la imagen
    |
    v
Dockle            <- inspeccion de la imagen construida (antes del push)
    |
    v
push al registro  <- solo si ambas validaciones pasan
```

- **Hadolint** actua sobre el texto del Dockerfile, sin necesitar Docker en ejecucion. Detecta malas practicas de escritura del archivo.
- **Dockle** actua sobre la imagen ya construida. Verifica que el resultado cumpla con criterios de seguridad en tiempo de ejecucion.

Ambas herramientas son complementarias: una valida la receta, la otra valida el producto.

## Hadolint: linter de Dockerfiles.

`Hadolint` analiza el Dockerfile de forma estatica antes de construir la imagen. Detecta instrucciones que violan las mejores practicas de seguridad y reproducibilidad: uso de `latest` como tag de imagen base, instalacion de paquetes sin versiones fijas, comandos `RUN` que acumulan capas innecesarias, o instrucciones que otorgan privilegios excesivos.

Comandos principales:

```bash
# instalar
brew install hadolint                     # macOS
docker pull hadolint/hadolint             # via Docker

# analizar el Dockerfile del proyecto
hadolint Dockerfile

# ignorar una regla especifica
hadolint --ignore DL3008 Dockerfile

# salir con error si hay hallazgos de nivel warning o superior
hadolint --failure-threshold warning Dockerfile
```

Hadolint tambien puede configurarse con un archivo `.hadolint.yaml` en la raiz del repositorio:

```yaml
ignore:
  - DL3008   # no exige versiones en apt-get (a veces aceptable en dev)
  - DL3009   # no limpia cache de apt en el mismo RUN
failure-threshold: error
```

En GitHub Actions existe una Action oficial que integra Hadolint directamente en el workflow sin necesidad de instalarlo manualmente.

## Dockle: inspeccion de la imagen construida.

Mientras Hadolint analiza el Dockerfile como texto, `Dockle` inspecciona la imagen ya construida. Verifica que la imagen resultante cumpla con practicas de seguridad definidas por CIS Benchmarks y otras guias:

- que no corra como `root`;
- que no tenga setuid/setgid en binarios;
- que use un usuario no privilegiado declarado en el Dockerfile;
- que no exponga credenciales en variables de entorno o capas de la imagen;
- que la imagen tenga un healthcheck definido.

Comandos principales:

```bash
# instalar
brew install goodwithtech/r/dockle        # macOS
docker pull goodwithtech/dockle           # via Docker

# inspeccionar imagen local
dockle mi-app:latest

# fallar si hay hallazgos WARN o superiores
dockle --exit-code 1 --exit-level warn mi-app:latest

# exportar reporte
dockle --format json --output dockle-result.json mi-app:latest
```

La distincion con Trivy es de proposito: Dockle verifica configuracion y buenas practicas de la imagen; Trivy busca CVEs en los paquetes instalados dentro de ella. Son complementarios y en un pipeline robusto corren los dos.

## Pipeline endurecido: flujo completo.

Combinando Hadolint y Dockle con el build y push, el job queda:

```yaml
jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Lint del Dockerfile con Hadolint
        uses: hadolint/hadolint-action@v3.1.0
        with:
          dockerfile: Dockerfile
          failure-threshold: warning

      - name: Construir imagen
        run: docker build -t mi-app:${{ github.sha }} .

      - name: Inspeccionar imagen con Dockle
        uses: erzz/dockle-action@v1
        with:
          image: mi-app:${{ github.sha }}
          exit-code: 1
          failure-threshold: WARN

      - name: Autenticar en registro
        uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}

      - name: Push al registro
        run: docker push ghcr.io/${{ github.repository }}:${{ github.sha }}
```

El orden importa: Hadolint corre sobre el archivo de texto antes del build (no requiere Docker); Dockle corre sobre la imagen ya construida pero antes del push. Ningun artefacto defectuoso llega al registro.

In [ ]:
version = '1.2.0'
sha_corta = 'abc1234'
tags = [
    f'ghcr.io/pythonista/py271-api:{version}',
    f'ghcr.io/pythonista/py271-api:{sha_corta}'
]

for tag in tags:
    print(tag)

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>